In [1]:
import pandas as pd

# 1. Load the Data
# Note: Ensure the Zillow filename matches what you uploaded!
airbnb = pd.read_csv('listings.csv.gz')
# We read the Zillow data. If your file has a different name, rename it here.
zillow = pd.read_csv('Neighborhood_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv')

# 2. Filter Zillow for New York City only
# Zillow data covers the whole US, so we must filter for NY to avoid errors
zillow = zillow[(zillow['City'] == 'New York') & (zillow['State'] == 'NY')]

# 3. Clean Airbnb Data
# Keep only the columns we need. 'neighbourhood_cleansed' is the Neighborhood Name
airbnb = airbnb[['id', 'neighbourhood_cleansed', 'price', 'room_type']]

# Remove $ and , from price so we can do math
airbnb['price'] = airbnb['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# Filter for 'Entire home/apt' (these impact housing supply the most)
airbnb = airbnb[airbnb['room_type'] == 'Entire home/apt']

# Count Airbnbs per Neighborhood
airbnb_counts = airbnb.groupby('neighbourhood_cleansed').size().reset_index(name='airbnb_count')
airbnb_counts.columns = ['neighborhood', 'airbnb_count']

# 4. Clean Zillow Data
# Get the latest price column (the last column in the file)
latest_price_col = zillow.columns[-1]

# Keep only the Neighborhood Name and the Latest Price
zillow_subset = zillow[['RegionName', latest_price_col]].copy()
zillow_subset.columns = ['neighborhood', 'avg_house_price']

print("Data cleaning complete!")
print(f"Found {len(airbnb_counts)} Airbnb neighborhoods and {len(zillow_subset)} Zillow neighborhoods.")

Data cleaning complete!
Found 221 Airbnb neighborhoods and 198 Zillow neighborhoods.


In [2]:
# Merge the tables on the 'neighborhood' name
master_df = pd.merge(airbnb_counts, zillow_subset, on='neighborhood', how='inner')

# Show the top 5 rows to verify
display(master_df.head())

,neighborhood,airbnb_count,avg_house_price
0,Arden Heights,3,540756.008687
1,Arrochar,10,763863.275120
2,Astoria,209,756030.032690
3,Bath Beach,15,802185.701580
4,Bay Ridge,33,804037.728698


In [3]:
import statsmodels.api as sm

# Define Y (Price) and X (Airbnb Count)
Y = master_df['avg_house_price']
X = master_df['airbnb_count']

# Add a 'constant' (the intercept) - required for OLS
X = sm.add_constant(X)

# Build and run the model
model = sm.OLS(Y, X).fit()

# Show the results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        avg_house_price   R-squared:                       0.097
Model:                            OLS   Adj. R-squared:                  0.092
Method:                 Least Squares   F-statistic:                     17.43
Date:                Tue, 23 Dec 2025   Prob (F-statistic):           4.86e-05
Time:                        17:09:55   Log-Likelihood:                -2350.7
No. Observations:                 164   AIC:                             4705.
Df Residuals:                     162   BIC:                             4712.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const         7.993e+05   3.52e+04     22.737   

In [7]:
import plotly.express as px

fig = px.scatter(master_df,
                 x="airbnb_count",
                 y="avg_house_price",
                 hover_name="neighborhood",
                 trendline="ols",  # Draws the regression line automatically
                 title="Correlation: Airbnb Density vs. Housing Prices in NYC",
                 labels={"airbnb_count": "Number of Airbnbs", "avg_house_price": "Average House Price ($)"})

fig.show()